# AIRL Policy Cross-Evaluation

Cross-evaluate the final AIRL time-GRU policy inside the PCNN-based simulation environment. If a compatible legacy LSTM checkpoint is available, the notebook will include it as an optional comparison run.


In [ ]:
import os
import sys
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display


def _iter_project_candidates(start: Path):
    seen = set()

    first_pass = [start]
    try:
        first_pass.extend(child for child in start.iterdir() if child.is_dir())
    except OSError:
        pass

    for candidate in first_pass:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        yield candidate

    for candidate in start.parents:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        yield candidate


def _locate_project_root(start: Path) -> Path:
    candidates = []
    for candidate in _iter_project_candidates(start):
        try:
            has_layout = (candidate / 'src').is_dir() and (candidate / 'configs').is_dir()
        except OSError:
            has_layout = False
        if not has_layout:
            continue

        score = 0
        if (candidate / 'notebooks' / 'README.md').is_file():
            score += 2
        if (candidate / 'tests' / 'test_standalone_repo.py').is_file():
            score += 2
        candidates.append((score, candidate))

    if not candidates:
        raise RuntimeError('Unable to locate project root from ' + str(start))

    best_score, best_candidate = candidates[0]
    for score, candidate in candidates[1:]:
        if score > best_score:
            best_score, best_candidate = score, candidate
    return best_candidate


try:
    NOTEBOOK_PATH = Path(__file__).resolve()
    NOTEBOOK_DIR = NOTEBOOK_PATH.parent
except NameError:
    NOTEBOOK_PATH = None
    NOTEBOOK_DIR = Path.cwd().resolve()

PROJECT_ROOT = _locate_project_root(NOTEBOOK_DIR)
if NOTEBOOK_DIR.name != 'notebooks' and (PROJECT_ROOT / 'notebooks').is_dir():
    NOTEBOOK_DIR = PROJECT_ROOT / 'notebooks'

if Path.cwd().resolve() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

src_path = str(PROJECT_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

plt.style.use('seaborn-v0_8')
print(f'Notebook path: {NOTEBOOK_PATH}')
print(f'Notebook directory: {NOTEBOOK_DIR}')
print(f'Project root: {PROJECT_ROOT}')
print(f"Using torch {torch.__version__}")

from configs.training_config import get_full_training_config
from data_processing import (
    load_and_filter_data,
    split_train_val,
    setup_scalers,
    set_random_seed,
    inverse_normalize,
    normalize,
    extract_expert_trajectories,
)
from models import load_dynamics_model, MMVPolicyActorCritic
from airl_environment import create_airl_environment
from rl_environment import MODE_MECH, MODE_MIXED, MODE_NATURAL, TEMP_LEVELS, FCU_ZONES
from rl_training import HybridPPOPolicy, masked_categorical


In [ ]:
config = get_full_training_config()
config.data.data_file = 'l14_merged_data_with_rain.csv'
config.data.min_trajectory_length = 690

# Final training notebook settings (kept for reference)
config.training.n_airl_iters = 20
config.training.batch_days = 16
config.training.val_days = 11
config.training.ppo_updates = 10
config.training.ppo_epochs = 6
config.training.policy_lr = 5e-4
config.training.reward_lr = 5e-4
config.training.reward_update_steps = 5

config.model.reward_type = 'temps_hold_dwell_prev_time_gru'
config.environment.comfort_max = 27.0
REPORT_COMFORT_CEILING = 30.0

set_random_seed(config.data.random_seed)

device = torch.device(config.training.device)
print(f"Device: {device}")
print(config)


In [ ]:
data_path = PROJECT_ROOT / config.data.data_file
print(f"Loading data from {data_path}")

daily_dataframes = load_and_filter_data(
    file_path=str(data_path),
    min_length=config.data.min_trajectory_length,
)
print(f"Total usable days: {len(daily_dataframes)}")


In [ ]:
# dates_to_exclude = {(9, 4), (10, 15), (7, 8), (10, 16), (8, 9), (10, 8), (10, 17), (7, 9)}

dates_to_exclude = {(7,17), (8,9)}

filtered_daily = []
removed_labels = []

for df in daily_dataframes:
    day = pd.to_datetime(df['date'].iloc[0])
    if (day.month, day.day) in dates_to_exclude:
        removed_labels.append(day.strftime('%Y-%m-%d'))
        continue
    filtered_daily.append(df)

daily_dataframes = filtered_daily
print(f"Removed {len(removed_labels)} days: {removed_labels}")
print(f"Remaining days: {len(daily_dataframes)}")

In [ ]:
train_dfs, val_dfs = split_train_val(
    daily_dataframes,
    split_ratio=config.data.train_val_split,
    shuffle=True,
    random_state=0
)
print(f"Train days: {len(train_dfs)} | Validation days: {len(val_dfs)}")

print("Setting up feature scalers...")
scalers = setup_scalers(train_dfs, val_dfs)
print(f"Created scalers for {len(scalers)} features")


In [ ]:
pcnn_model_path = PROJECT_ROOT / config.dynamics_model_path
print(f"Loading PCNN dynamics from {pcnn_model_path}")

pcnn_dynamics = load_dynamics_model(
    model_path=str(pcnn_model_path),
    device=torch.device('cpu'),
    hidden_dim=config.model.dynamics_hidden_dim,
    output_dim=config.model.dynamics_output_dim,
    ac_dim=config.model.dynamics_ac_channels,
    nv_dim=config.model.dynamics_nv_channels,
)
pcnn_dynamics.eval()

eval_env_kwargs = dict(
    daily_data_list=val_dfs,
    dynamics_model=pcnn_dynamics,
    scalers=scalers,
    num_zones=config.model.num_zones,
    comfort_min=config.environment.comfort_min,
    comfort_max=REPORT_COMFORT_CEILING,
    heavy_wind_threshold=config.environment.heavy_wind_threshold,
    policy_time_features=True,
)

eval_env = create_airl_environment(**eval_env_kwargs)
initial_state = eval_env.reset(day_index=0)
state_dim = initial_state.shape[0]
print(f"State dimension: {state_dim}")
print(f"Validation horizon (steps): {eval_env.max_steps}")


In [ ]:
output_dir = PROJECT_ROOT / config.output_dir
print(f"Loading policies from {output_dir}")

reward_tag = config.model.reward_type
PRIMARY_POLICY_NAME = 'AIRL-Time-GRU' if reward_tag == 'temps_hold_dwell_prev_time_gru' else reward_tag.replace('_', '-')
COMPARISON_POLICY_NAME = 'Legacy LSTM'
PRIMARY_EXEC_LABEL = f'{PRIMARY_POLICY_NAME} policy on PCNN env'
COMPARISON_EXEC_LABEL = f'{COMPARISON_POLICY_NAME} policy on PCNN env'

primary_policy_path = output_dir / f'airl_mode_policy_{reward_tag}_final.pth'
comparison_policy_path = output_dir / 'airl_mode_policy_lstm_final.pth'

if not primary_policy_path.exists():
    raise FileNotFoundError(f'Primary policy checkpoint not found: {primary_policy_path}')


def _load_policy(path: Path) -> Union[HybridPPOPolicy, MMVPolicyActorCritic]:
    state_dict = torch.load(path, map_location=device)
    if any(key.startswith('trunk.model') for key in state_dict.keys()):
        policy = HybridPPOPolicy(obs_dim=state_dim).to(device)
    elif 'change_head.weight' in state_dict:
        policy = MMVPolicyActorCritic(state_dim=state_dim, num_zones=config.model.num_zones).to(device)
    else:
        raise RuntimeError(f'Unrecognized policy checkpoint format: {path}')

    incompatible = policy.load_state_dict(state_dict, strict=False)
    if isinstance(incompatible, tuple):
        missing, unexpected = incompatible
    else:
        missing, unexpected = incompatible.missing_keys, incompatible.unexpected_keys

    if missing:
        print(f"  Warning: missing keys when loading {path.name}: {missing}")
    if unexpected:
        print(f"  Warning: unexpected keys when loading {path.name}: {unexpected}")

    policy.eval()
    return policy


primary_policy = _load_policy(primary_policy_path)
print(f"Loaded {PRIMARY_POLICY_NAME} policy from {primary_policy_path.name}")

comparison_policy = None
HAS_COMPARISON_POLICY = False
if comparison_policy_path.exists():
    try:
        comparison_policy = _load_policy(comparison_policy_path)
        HAS_COMPARISON_POLICY = True
        print(f"Loaded {COMPARISON_POLICY_NAME} policy from {comparison_policy_path.name}")
    except Exception as exc:
        print(f"Skipping legacy LSTM comparison: {exc}")
else:
    print('Legacy LSTM checkpoint not found; running single-policy evaluation.')


In [ ]:
from typing import Sequence

def evaluate_policy_on_env(
    env_factory: Callable[[], Any],
    exec_policy: Union[HybridPPOPolicy, MMVPolicyActorCritic],
    exec_label: str,
    day_indices: Iterable[int],
    device: torch.device,
    ref_policy: Optional[Union[HybridPPOPolicy, MMVPolicyActorCritic]] = None,
    ref_label: Optional[str] = None,
) -> List[Dict[str, Any]]:
    """Run a legacy AIRL policy in the evaluation environment while logging metrics."""
    env = env_factory()
    results: List[Dict[str, Any]] = []

    zone_cols = env.columns.get('zone_cols', [])
    supply_cols = env.columns.get('fcu_supply_cols', [])
    lc_cols = env.columns.get('lc_cols', [])
    scalers = env.scalers

    def _denormalize_sequence(values: Sequence[float], columns: Sequence[str]) -> List[float]:
        return [float(inverse_normalize(float(values[i]), col, scalers)) if i < len(values) else float('nan')
                for i, col in enumerate(columns)]

    for day_idx in day_indices:
        state = env.reset(day_index=day_idx)
        done = False

        energy_series: List[float] = []
        comfort_series: List[float] = []
        exec_changes: List[int] = []
        ref_changes: List[int] = [] if ref_policy is not None else []

        zone_temp_series: List[List[float]] = []
        supply_temp_series: List[List[float]] = []
        local_cooling_series: List[List[float]] = []
        outdoor_temp_series: List[float] = []
        window_state_series: List[int] = []
        time_index: List[int] = []

        window_matches = supply_matches = local_matches = 0
        window_comparisons = supply_comparisons = local_comparisons = 0
        steps = 0

        while not done:
            if zone_cols:
                zone_temp_series.append(_denormalize_sequence(state[:len(zone_cols)], zone_cols))
            else:
                zone_temp_series.append([])

            state_tensor = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

            def _policy_step(policy):
                if hasattr(policy, 'act'):
                    action_dict, _, _ = policy.act(state_tensor)
                    return {
                        'change': int(action_dict['change']),
                        'local_cooling': np.asarray(action_dict['local_cooling'], dtype=np.float32),
                        'supply_temps': np.asarray(action_dict['supply_temps'], dtype=np.float32),
                    }
                raise RuntimeError('Policy does not support legacy act() interface')

            exec_action = _policy_step(exec_policy)
            ref_action = _policy_step(ref_policy) if ref_policy is not None else None

            next_state, _, done, info = env.step(exec_action)

            energy_series.append(float(info.get('energy_consumption', 0.0)))
            comfort_series.append(float(info.get('comfort_violations', 0.0)))
            exec_changes.append(int(exec_action['change']))
            window_state_series.append(int(env.prev_window))
            outdoor_temp_series.append(float(info.get('outdoor_temperature', float('nan'))))
            time_index.append(steps)

            if supply_cols:
                supply_temp_series.append([
                    float(inverse_normalize(env.prev_supply_temps[i], supply_cols[i], scalers))
                    for i in range(min(len(supply_cols), len(env.prev_supply_temps)))
                ])
            else:
                supply_temp_series.append([])
            if lc_cols:
                local_cooling_series.append([
                    float(inverse_normalize(env.prev_lc_temps[i], lc_cols[i], scalers))
                    for i in range(min(len(lc_cols), len(env.prev_lc_temps)))
                ])
            else:
                local_cooling_series.append([])

            if ref_action is not None:
                ref_changes.append(int(ref_action['change']))
                window_matches += int(ref_action['change'] == exec_action['change'])
                window_comparisons += 1

                supply_matches += int(np.allclose(exec_action['supply_temps'], ref_action['supply_temps'], atol=1e-2))
                supply_comparisons += 1

                local_matches += int(np.allclose(exec_action['local_cooling'], ref_action['local_cooling'], atol=1e-2))
                local_comparisons += 1

            state = next_state
            steps += 1

        exec_changes_arr = np.asarray(exec_changes, dtype=np.float32)
        ref_changes_arr = np.asarray(ref_changes, dtype=np.float32) if ref_policy is not None else np.asarray([], dtype=np.float32)

        window_agreement_rate = (window_matches / window_comparisons) if window_comparisons > 0 else float('nan')
        supply_agreement_rate = (supply_matches / supply_comparisons) if supply_comparisons > 0 else float('nan')
        local_agreement_rate = (local_matches / local_comparisons) if local_comparisons > 0 else float('nan')
        if (
            ref_policy is not None and ref_changes_arr.size > 1
            and np.var(exec_changes_arr) > 0 and np.var(ref_changes_arr) > 0
        ):
            window_action_corr = float(np.corrcoef(exec_changes_arr, ref_changes_arr)[0, 1])
        else:
            window_action_corr = float('nan')

        results.append({
            'policy_label': exec_label,
            'reference_policy': ref_label if ref_label is not None else None,
            'day_index': day_idx,
            'energy_kwh': float(np.sum(energy_series)),
            'comfort_degC_min': float(np.sum(comfort_series)),
            'window_switches': int(np.sum(exec_changes_arr)),
            'episode_length': steps,
            'window_agreement_rate': window_agreement_rate,
            'supply_temp_agreement_rate': supply_agreement_rate,
            'local_cooling_agreement_rate': local_agreement_rate,
            'window_matches': window_matches,
            'window_comparisons': window_comparisons,
            'supply_matches': supply_matches,
            'supply_comparisons': supply_comparisons,
            'local_matches': local_matches,
            'local_comparisons': local_comparisons,
            'window_action_corr': window_action_corr,
            'exec_action_seq': exec_changes if ref_policy is not None else None,
            'ref_action_seq': ref_changes if ref_policy is not None else None,
            'zone_temp_seq': zone_temp_series,
            'supply_temp_seq': supply_temp_series,
            'local_cooling_seq': local_cooling_series,
            'window_state_seq': window_state_series,
            'outdoor_temp_seq': outdoor_temp_series,
            'time_index': time_index,
        })

    return results



In [ ]:
val_day_indices = range(len(val_dfs))

make_eval_env = lambda: create_airl_environment(**eval_env_kwargs)

primary_exec_results = evaluate_policy_on_env(
    env_factory=make_eval_env,
    exec_policy=primary_policy,
    exec_label=PRIMARY_EXEC_LABEL,
    day_indices=val_day_indices,
    device=device,
)

combined_results = list(primary_exec_results)
if HAS_COMPARISON_POLICY:
    comparison_exec_results = evaluate_policy_on_env(
        env_factory=make_eval_env,
        exec_policy=comparison_policy,
        exec_label=COMPARISON_EXEC_LABEL,
        day_indices=val_day_indices,
        device=device,
        ref_policy=primary_policy,
        ref_label=f'Reference: {PRIMARY_EXEC_LABEL}',
    )
    combined_results.extend(comparison_exec_results)

results_df = pd.DataFrame(combined_results)

preview_cols = [
    'policy_label',
    'reference_policy',
    'day_index',
    'energy_kwh',
    'comfort_degC_min',
    'window_switches',
    'window_agreement_rate',
    'supply_temp_agreement_rate',
    'local_cooling_agreement_rate',
]
print("Combined evaluation results (first five rows):")
display(results_df[preview_cols].head())


In [ ]:
metric_cols = [
    'energy_kwh',
    'comfort_degC_min',
    'window_switches',
    'window_agreement_rate',
    'supply_temp_agreement_rate',
    'local_cooling_agreement_rate',
    'window_action_corr',
]

summary_stats = (
    results_df.groupby('policy_label')[metric_cols]
    .agg(['mean', 'std'])
    .sort_index()
)
print("Per-policy summary statistics:")
display(summary_stats)



In [ ]:
if not HAS_COMPARISON_POLICY:
    print('Only the final GRU policy is available; showing day-level metrics for that run.')
    primary_day_table = (
        results_df[results_df['policy_label'] == PRIMARY_EXEC_LABEL]
        [['day_index', 'energy_kwh', 'comfort_degC_min', 'window_switches']]
        .sort_values('day_index')
        .reset_index(drop=True)
    )
    display(primary_day_table)
else:
    comparison_table = (
        results_df
        .set_index(['day_index', 'policy_label'])
        [['energy_kwh', 'comfort_degC_min', 'window_switches']]
        .unstack('policy_label')
    )
    comparison_table.columns = [' '.join(col).strip() for col in comparison_table.columns]

    comparison_table['energy_delta (comparison-primary)'] = (
        comparison_table[f'energy_kwh {COMPARISON_EXEC_LABEL}']
        - comparison_table[f'energy_kwh {PRIMARY_EXEC_LABEL}']
    )
    comparison_table['comfort_delta (comparison-primary)'] = (
        comparison_table[f'comfort_degC_min {COMPARISON_EXEC_LABEL}']
        - comparison_table[f'comfort_degC_min {PRIMARY_EXEC_LABEL}']
    )
    comparison_table['window_switch_delta (comparison-primary)'] = (
        comparison_table[f'window_switches {COMPARISON_EXEC_LABEL}']
        - comparison_table[f'window_switches {PRIMARY_EXEC_LABEL}']
    )

    print('Day-level comparison of core metrics:')
    display(comparison_table)


In [ ]:
from matplotlib.patches import Patch

if not HAS_COMPARISON_POLICY:
    print('No compatible secondary policy loaded; skipping policy-vs-policy trace plots.')
else:
    primary_rows = (
        results_df[results_df['policy_label'] == PRIMARY_EXEC_LABEL]
        .sort_values('day_index')
        .set_index('day_index')
    )
    comparison_rows = (
        results_df[results_df['policy_label'] == COMPARISON_EXEC_LABEL]
        .sort_values('day_index')
        .set_index('day_index')
    )

    zone_labels = [col.replace('_', ' ') for col in eval_env.columns['zone_cols'][:config.model.num_zones]]
    supply_labels = [col.replace('_', ' ') for col in eval_env.columns['fcu_supply_cols'][:config.model.num_zones]]
    local_labels = [col.replace('_', ' ') for col in eval_env.columns['lc_cols']]

    common_days = sorted(set(primary_rows.index).intersection(comparison_rows.index))
    cmap = plt.cm.get_cmap('tab10')

    shade_handles = [
        Patch(facecolor='steelblue', alpha=0.08, label=f'Window open ({PRIMARY_POLICY_NAME})'),
        Patch(facecolor='darkorange', alpha=0.08, label=f'Window open ({COMPARISON_POLICY_NAME})'),
    ]

    def add_window_shading(ax, time_vals: np.ndarray, states: np.ndarray, color: str, alpha: float) -> None:
        if time_vals.size == 0 or states.size == 0:
            return
        start = None
        for idx, val in enumerate(states):
            is_open = val >= 0.5
            if is_open and start is None:
                start = time_vals[idx]
            elif not is_open and start is not None:
                end = time_vals[idx] + 1
                ax.axvspan(start, end, color=color, alpha=alpha, zorder=0)
                start = None
        if start is not None:
            ax.axvspan(start, time_vals[-1] + 1, color=color, alpha=alpha, zorder=0)

    for day_idx in common_days:
        primary_row = primary_rows.loc[day_idx]
        comparison_row = comparison_rows.loc[day_idx]

        zone_primary = np.asarray(primary_row['zone_temp_seq'], dtype=np.float32)
        zone_comparison = np.asarray(comparison_row['zone_temp_seq'], dtype=np.float32)
        supply_primary = np.asarray(primary_row['supply_temp_seq'], dtype=np.float32)
        supply_comparison = np.asarray(comparison_row['supply_temp_seq'], dtype=np.float32)
        local_primary = np.asarray(primary_row['local_cooling_seq'], dtype=np.float32)
        local_comparison = np.asarray(comparison_row['local_cooling_seq'], dtype=np.float32)
        window_primary = np.asarray(primary_row['window_state_seq'], dtype=np.float32)
        window_comparison = np.asarray(comparison_row['window_state_seq'], dtype=np.float32)
        time_primary = np.asarray(primary_row['time_index'], dtype=np.float32)
        time_comparison = np.asarray(comparison_row['time_index'], dtype=np.float32)

        zone_primary = np.atleast_2d(zone_primary)
        zone_comparison = np.atleast_2d(zone_comparison)
        supply_primary = np.atleast_2d(supply_primary)
        supply_comparison = np.atleast_2d(supply_comparison)
        local_primary = np.atleast_2d(local_primary)
        local_comparison = np.atleast_2d(local_comparison)

        fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
        fig.suptitle(f'Day {day_idx} Policy Comparison', fontsize=14)

        for ax in axes:
            add_window_shading(ax, time_primary, window_primary, color='steelblue', alpha=0.08)
            add_window_shading(ax, time_comparison, window_comparison, color='darkorange', alpha=0.08)

        for idx, label in enumerate(zone_labels):
            color = cmap(idx % 10)
            if zone_primary.shape[1] > idx:
                axes[0].plot(time_primary, zone_primary[:, idx], color=color, linestyle='-', label=f'{PRIMARY_POLICY_NAME} {label}')
            if zone_comparison.shape[1] > idx:
                axes[0].plot(time_comparison, zone_comparison[:, idx], color=color, linestyle='--', label=f'{COMPARISON_POLICY_NAME} {label}')
        axes[0].set_ylabel('Zone Temperature (°C)')
        handles, labels = axes[0].get_legend_handles_labels()
        axes[0].legend(handles + shade_handles, labels + [h.get_label() for h in shade_handles], loc='upper right', fontsize=8, ncol=2)
        axes[0].grid(True, alpha=0.3)

        for idx, label in enumerate(supply_labels):
            color = cmap(idx % 10)
            if supply_primary.shape[1] > idx:
                axes[1].plot(time_primary, supply_primary[:, idx], color=color, linestyle='-', label=f'{PRIMARY_POLICY_NAME} {label}')
            if supply_comparison.shape[1] > idx:
                axes[1].plot(time_comparison, supply_comparison[:, idx], color=color, linestyle='--', label=f'{COMPARISON_POLICY_NAME} {label}')
        axes[1].set_ylabel('FCU Supply Temp (°C)')
        handles, labels = axes[1].get_legend_handles_labels()
        axes[1].legend(handles, labels, loc='upper right', fontsize=8, ncol=2)
        axes[1].grid(True, alpha=0.3)

        if local_primary.size > 0 or local_comparison.size > 0:
            for idx, label in enumerate(local_labels):
                color = cmap((idx + len(supply_labels)) % 10)
                if local_primary.shape[1] > idx:
                    axes[2].plot(time_primary, local_primary[:, idx], color=color, linestyle='-', label=f'{PRIMARY_POLICY_NAME} {label}')
                if local_comparison.shape[1] > idx:
                    axes[2].plot(time_comparison, local_comparison[:, idx], color=color, linestyle='--', label=f'{COMPARISON_POLICY_NAME} {label}')
        axes[2].set_ylabel('Local Cooling Temp (°C)')
        axes[2].set_xlabel('Time Step (minutes)')
        handles, labels = axes[2].get_legend_handles_labels()
        axes[2].legend(handles + shade_handles, labels + [h.get_label() for h in shade_handles], loc='upper right', fontsize=8, ncol=2)
        axes[2].grid(True, alpha=0.3)

        plt.tight_layout(rect=[0, 0, 1, 0.97])
        plt.show()


In [ ]:
if not HAS_COMPARISON_POLICY:
    print('No compatible secondary policy loaded; skipping overall policy agreement summary.')
else:
    primary_rows = results_df[results_df['policy_label'] == PRIMARY_EXEC_LABEL].set_index('day_index')
    comparison_rows = results_df[results_df['policy_label'] == COMPARISON_EXEC_LABEL].set_index('day_index')

    common_days = sorted(set(primary_rows.index).intersection(comparison_rows.index))
    if not common_days:
        raise RuntimeError('No overlapping evaluation days between the primary and comparison runs')

    window_matches = 0
    window_comparisons = 0
    supply_matches = 0
    supply_comparisons = 0
    local_matches = 0
    local_comparisons = 0
    window_correlations = []

    for day in common_days:
        primary_entry = primary_rows.loc[day]
        comparison_entry = comparison_rows.loc[day]

        win_primary = np.asarray(primary_entry['window_state_seq'], dtype=np.float32)
        win_comparison = np.asarray(comparison_entry['window_state_seq'], dtype=np.float32)
        supply_primary = np.asarray(primary_entry['supply_temp_seq'], dtype=np.float32)
        supply_comparison = np.asarray(comparison_entry['supply_temp_seq'], dtype=np.float32)
        local_primary = np.asarray(primary_entry['local_cooling_seq'], dtype=np.float32)
        local_comparison = np.asarray(comparison_entry['local_cooling_seq'], dtype=np.float32)

        min_len = min(len(win_primary), len(win_comparison))
        if min_len == 0:
            continue

        win_primary = win_primary[:min_len]
        win_comparison = win_comparison[:min_len]
        window_matches += int(np.sum(np.isclose(win_primary, win_comparison, atol=0.0)))
        window_comparisons += min_len

        if supply_primary.size and supply_comparison.size:
            min_steps = min(supply_primary.shape[0], supply_comparison.shape[0])
            min_cols = min(supply_primary.shape[1], supply_comparison.shape[1])
            diff = np.abs(supply_primary[:min_steps, :min_cols] - supply_comparison[:min_steps, :min_cols])
            supply_matches += int(np.sum(diff <= 1.0))
            supply_comparisons += diff.size

        if local_primary.size and local_comparison.size:
            min_steps = min(local_primary.shape[0], local_comparison.shape[0])
            min_cols = min(local_primary.shape[1], local_comparison.shape[1])
            diff = np.abs(local_primary[:min_steps, :min_cols] - local_comparison[:min_steps, :min_cols])
            local_matches += int(np.sum(diff <= 1.0))
            local_comparisons += diff.size

        exec_changes = np.diff(win_comparison, prepend=win_comparison[0])
        ref_changes = np.diff(win_primary, prepend=win_primary[0])
        if exec_changes.size > 1 and np.var(exec_changes) > 0 and np.var(ref_changes) > 0:
            window_correlations.append(float(np.corrcoef(exec_changes, ref_changes)[0, 1]))

    overall_window_agreement = window_matches / window_comparisons if window_comparisons else float('nan')
    overall_supply_agreement = supply_matches / supply_comparisons if supply_comparisons else float('nan')
    overall_local_agreement = local_matches / local_comparisons if local_comparisons else float('nan')
    overall_window_corr = float(np.nanmean(window_correlations)) if window_correlations else float('nan')

    print(f"Overall window agreement ({COMPARISON_POLICY_NAME} vs {PRIMARY_POLICY_NAME}): {overall_window_agreement:.3%}")
    print(f"Overall supply temperature agreement (<=1°C difference): {overall_supply_agreement:.3%}")
    print(f"Overall local cooling agreement (<=1°C difference): {overall_local_agreement:.3%}")
    print(f"Overall window action correlation: {overall_window_corr:.3f}")


In [ ]:
FCU_ON_THRESHOLD = 18.0  # °C threshold for FCU cooling being considered on

if 'compute_fcu_on' not in globals():
    def compute_fcu_on(supply_seq, window_seq, threshold):
        supply = np.asarray(supply_seq, dtype=np.float32)
        if supply.size == 0:
            return np.zeros((0, 0), dtype=bool)
        supply = np.atleast_2d(supply)
        window = np.asarray(window_seq, dtype=np.float32).reshape(-1)
        steps = min(supply.shape[0], window.shape[0])
        if steps == 0:
            return np.zeros((0, supply.shape[1]), dtype=bool)
        supply = supply[:steps]
        closed = window[:steps] < 0.5
        return (supply < threshold) & closed[:, None]

    def compute_local_on(local_seq, outdoor_seq):
        local = np.asarray(local_seq, dtype=np.float32)
        if local.size == 0:
            return np.zeros((0, 0), dtype=bool)
        local = np.atleast_2d(local)
        outdoor = np.asarray(outdoor_seq, dtype=np.float32).reshape(-1)
        steps = min(local.shape[0], outdoor.shape[0])
        if steps == 0:
            return np.zeros((0, local.shape[1]), dtype=bool)
        local = local[:steps]
        outdoor = outdoor[:steps]
        return local < outdoor[:, None]

    def count_agreement(a, b):
        if a.size == 0 or b.size == 0:
            return 0, 0
        steps = min(a.shape[0], b.shape[0])
        cols = min(a.shape[1], b.shape[1])
        if steps == 0 or cols == 0:
            return 0, 0
        return int(np.sum(a[:steps, :cols] == b[:steps, :cols])), steps * cols

if 'expert_reference_df' not in globals():
    expert_entries = []
    zone_cols = eval_env.columns['zone_cols'][:config.model.num_zones]
    supply_cols = eval_env.columns['fcu_supply_cols'][:config.model.num_zones]
    lc_cols = eval_env.columns['lc_cols']
    expert_trajs = extract_expert_trajectories(
        val_dfs,
        scalers,
        window_col='Z1 Windows Open Close Status',
        comfort_max=REPORT_COMFORT_CEILING,
        comfort_min=config.environment.comfort_min,
    )

    def _denorm_rows(norm_rows, cols):
        if not cols:
            return [[] for _ in range(len(norm_rows))]
        return [
            [float(inverse_normalize(val, cols[i], scalers)) for i, val in enumerate(row[:len(cols)])]
            for row in norm_rows
        ]

    for day_idx, traj in enumerate(expert_trajs):
        states = np.asarray(traj['states'], dtype=np.float32)
        actions = np.asarray(traj['actions'], dtype=np.float32)
        if states.size == 0 or actions.size == 0:
            continue
        window_seq = (states[:, -1] >= 0.5).astype(np.int32)
        local_norm = actions[:, 1:1 + len(lc_cols)] if len(lc_cols) else np.zeros((len(window_seq), 0), dtype=np.float32)
        supply_start = 1 + len(lc_cols)
        supply_norm = actions[:, supply_start:supply_start + len(supply_cols)] if len(supply_cols) else np.zeros((len(window_seq), 0), dtype=np.float32)
        outdoor_norm = states[:, len(zone_cols)] if len(zone_cols) < states.shape[1] else np.zeros(len(window_seq), dtype=np.float32)
        outdoor_seq = [
            float(inverse_normalize(val, 'OutdoorTemperatureWindow', scalers))
            for val in outdoor_norm
        ]
        expert_entries.append({
            'policy_label': 'Expert Demonstration',
            'day_index': day_idx,
            'window_state_seq': window_seq.tolist(),
            'supply_temp_seq': _denorm_rows(supply_norm, supply_cols),
            'local_cooling_seq': _denorm_rows(local_norm, lc_cols),
            'outdoor_temp_seq': outdoor_seq,
            'time_index': list(range(len(window_seq))),
        })

    expert_reference_df = pd.DataFrame(expert_entries)
    print(f"Prepared expert reference sequences for {len(expert_reference_df)} validation days")

if expert_reference_df.empty:
    raise RuntimeError('No expert reference trajectories available for comparison')

expert_rows = expert_reference_df.set_index('day_index')
expert_summary_rows = []

for policy_label in results_df['policy_label'].dropna().unique():
    policy_rows = results_df[results_df['policy_label'] == policy_label].set_index('day_index')
    common_days = sorted(set(expert_rows.index).intersection(policy_rows.index))
    if not common_days:
        continue

    window_matches = window_comparisons = 0
    fcu_matches = fcu_comparisons = 0
    local_matches = local_comparisons = 0
    window_correlations = []

    for day in common_days:
        expert_entry = expert_rows.loc[day]
        policy_entry = policy_rows.loc[day]

        win_expert = np.asarray(expert_entry['window_state_seq'], dtype=np.float32)
        win_policy = np.asarray(policy_entry['window_state_seq'], dtype=np.float32)
        min_len = min(len(win_expert), len(win_policy))
        if min_len == 0:
            continue
        window_matches += int(np.sum(np.isclose(win_expert[:min_len], win_policy[:min_len], atol=0.0)))
        window_comparisons += min_len

        supply_expert = np.asarray(expert_entry['supply_temp_seq'], dtype=np.float32)
        supply_policy = np.asarray(policy_entry['supply_temp_seq'], dtype=np.float32)
        fcu_expert = compute_fcu_on(supply_expert, win_expert, FCU_ON_THRESHOLD)
        fcu_policy = compute_fcu_on(supply_policy, win_policy, FCU_ON_THRESHOLD)
        matches, comps = count_agreement(fcu_expert, fcu_policy)
        fcu_matches += matches
        fcu_comparisons += comps

        local_expert = np.asarray(expert_entry['local_cooling_seq'], dtype=np.float32)
        local_policy = np.asarray(policy_entry['local_cooling_seq'], dtype=np.float32)
        local_steps = min(len(local_expert), len(local_policy))
        if local_steps > 0:
            local_zones = min(
                local_expert.shape[1] if local_expert.ndim == 2 else 0,
                local_policy.shape[1] if local_policy.ndim == 2 else 0
            )
            if local_zones > 0:
                local_diff = np.abs(local_expert[:local_steps, :local_zones] - local_policy[:local_steps, :local_zones])
                local_matches += int(np.sum(local_diff < 3.0))
                local_comparisons += local_steps * local_zones

        exec_changes = np.diff(win_policy[:min_len], prepend=win_policy[0])
        ref_changes = np.diff(win_expert[:min_len], prepend=win_expert[0])
        if exec_changes.size > 1 and np.var(exec_changes) > 0 and np.var(ref_changes) > 0:
            window_correlations.append(float(np.corrcoef(exec_changes, ref_changes)[0, 1]))

    expert_summary_rows.append({
        'policy_label': policy_label,
        'window_agreement_rate': window_matches / window_comparisons if window_comparisons else float('nan'),
        'fcu_agreement_rate': fcu_matches / fcu_comparisons if fcu_comparisons else float('nan'),
        'local_agreement_rate': local_matches / local_comparisons if local_comparisons else float('nan'),
        'window_action_corr': float(np.nanmean(window_correlations)) if window_correlations else float('nan'),
        'n_days': len(common_days),
    })

expert_comparison_df = pd.DataFrame(expert_summary_rows).sort_values('policy_label').reset_index(drop=True)
print('Policy vs expert agreement summary:')
display(expert_comparison_df)


## Policy vs Expert Action Traces


In [ ]:
primary_rows = results_df[results_df['policy_label'] == PRIMARY_EXEC_LABEL].set_index('day_index')
plot_days = sorted(set(expert_rows.index).intersection(primary_rows.index))
if not plot_days:
    print('No overlapping days between the primary policy evaluation and expert reference for plotting.')
else:
    max_days = min(3, len(plot_days))
    supply_labels = [col.replace('_', ' ') for col in eval_env.columns['fcu_supply_cols'][:config.model.num_zones]]
    local_labels = [col.replace('_', ' ') for col in eval_env.columns['lc_cols']]

    for day in plot_days[:max_days]:
        expert_entry = expert_rows.loc[day]
        primary_entry = primary_rows.loc[day]

        win_policy = np.asarray(primary_entry['window_state_seq'], dtype=np.float32)
        win_expert = np.asarray(expert_entry['window_state_seq'], dtype=np.float32)
        t_policy = np.arange(len(win_policy))
        t_expert = np.arange(len(win_expert))

        supply_policy = np.asarray(primary_entry['supply_temp_seq'], dtype=np.float32)
        supply_expert = np.asarray(expert_entry['supply_temp_seq'], dtype=np.float32)
        local_policy = np.asarray(primary_entry['local_cooling_seq'], dtype=np.float32)
        local_expert = np.asarray(expert_entry['local_cooling_seq'], dtype=np.float32)

        fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=False)
        fig.suptitle(f'Day {day} | {PRIMARY_POLICY_NAME} policy vs Expert demonstration')

        axes[0].step(t_policy, win_policy, where='post', label=f'{PRIMARY_POLICY_NAME} window', color='tab:blue')
        axes[0].step(t_expert, win_expert, where='post', label='Expert window', color='tab:orange', linestyle='--')
        axes[0].set_ylabel('Window state')
        axes[0].set_ylim(-0.2, 1.2)
        axes[0].legend(loc='upper right')
        axes[0].grid(alpha=0.3, linestyle='--')

        ax_supply = axes[1]
        if supply_policy.size and supply_expert.size:
            steps = min(len(supply_policy), len(supply_expert))
            zones = min(supply_policy.shape[1], supply_expert.shape[1]) if supply_policy.ndim == 2 and supply_expert.ndim == 2 else 0
            time_axis = np.arange(steps)
            if zones == 0:
                ax_supply.text(0.5, 0.5, 'No FCU supply data', ha='center', va='center')
            else:
                for z in range(zones):
                    label = supply_labels[z] if z < len(supply_labels) else f'FCU {z + 1}'
                    ax_supply.plot(time_axis, supply_policy[:steps, z], label=f'{PRIMARY_POLICY_NAME} {label}')
                    ax_supply.plot(time_axis, supply_expert[:steps, z], linestyle='--', label=f'Expert {label}')
                ax_supply.legend(loc='upper right', ncol=2, fontsize='small')
        else:
            ax_supply.text(0.5, 0.5, 'No FCU supply data', ha='center', va='center')
        ax_supply.set_ylabel('Supply temp (°C)')
        ax_supply.grid(alpha=0.3, linestyle='--')

        ax_local = axes[2]
        if local_policy.size and local_expert.size:
            steps = min(len(local_policy), len(local_expert))
            zones = min(local_policy.shape[1], local_expert.shape[1]) if local_policy.ndim == 2 and local_expert.ndim == 2 else 0
            time_axis = np.arange(steps)
            if zones == 0:
                ax_local.text(0.5, 0.5, 'No local cooling data', ha='center', va='center')
            else:
                for z in range(zones):
                    label = local_labels[z] if z < len(local_labels) else f'LC {z + 1}'
                    ax_local.plot(time_axis, local_policy[:steps, z], label=f'{PRIMARY_POLICY_NAME} {label}')
                    ax_local.plot(time_axis, local_expert[:steps, z], linestyle='--', label=f'Expert {label}')
                ax_local.legend(loc='upper right', ncol=2, fontsize='small')
        else:
            ax_local.text(0.5, 0.5, 'No local cooling data', ha='center', va='center')
        ax_local.set_ylabel('Local cooling target (°C)')
        ax_local.set_xlabel('Step')
        ax_local.grid(alpha=0.3, linestyle='--')

        plt.tight_layout()
        plt.show()


In [ ]:
primary_summary = expert_comparison_df[expert_comparison_df['policy_label'] == PRIMARY_EXEC_LABEL]
if primary_summary.empty:
    raise RuntimeError('Primary policy summary missing from expert comparison results')

primary_summary = primary_summary.iloc[0]
print(f"Overall window agreement ({PRIMARY_POLICY_NAME} vs Expert): {primary_summary['window_agreement_rate']:.3%}")
print(f"Overall FCU on/off agreement: {primary_summary['fcu_agreement_rate']:.3%}")
print(f"Overall local cooling agreement: {primary_summary['local_agreement_rate']:.3%}")
print(f"Overall window action correlation: {primary_summary['window_action_corr']:.3f}")


In [ ]:
if not HAS_COMPARISON_POLICY:
    print('No compatible secondary policy loaded; skipping FCU/local on-off agreement summary.')
else:
    primary_rows = results_df[results_df['policy_label'] == PRIMARY_EXEC_LABEL].set_index('day_index')
    comparison_rows = results_df[results_df['policy_label'] == COMPARISON_EXEC_LABEL].set_index('day_index')

    common_days = sorted(set(primary_rows.index).intersection(comparison_rows.index))
    if not common_days:
        raise RuntimeError('No overlapping evaluation days between the primary and comparison runs')

    window_matches = window_comparisons = 0
    fcu_matches = fcu_comparisons = 0
    local_matches = local_comparisons = 0
    window_correlations = []

    for day in common_days:
        primary_entry = primary_rows.loc[day]
        comparison_entry = comparison_rows.loc[day]

        win_primary = np.asarray(primary_entry['window_state_seq'], dtype=np.float32)
        win_comparison = np.asarray(comparison_entry['window_state_seq'], dtype=np.float32)
        min_len = min(len(win_primary), len(win_comparison))
        if min_len == 0:
            continue
        window_matches += int(np.sum(np.isclose(win_primary[:min_len], win_comparison[:min_len], atol=0.0)))
        window_comparisons += min_len

        supply_primary = np.asarray(primary_entry['supply_temp_seq'], dtype=np.float32)
        supply_comparison = np.asarray(comparison_entry['supply_temp_seq'], dtype=np.float32)
        fcu_primary = compute_fcu_on(supply_primary, win_primary, FCU_ON_THRESHOLD)
        fcu_comparison = compute_fcu_on(supply_comparison, win_comparison, FCU_ON_THRESHOLD)
        matches, comps = count_agreement(fcu_primary, fcu_comparison)
        fcu_matches += matches
        fcu_comparisons += comps

        local_primary = np.asarray(primary_entry['local_cooling_seq'], dtype=np.float32)
        local_comparison = np.asarray(comparison_entry['local_cooling_seq'], dtype=np.float32)
        outdoor_primary = np.asarray(primary_entry['outdoor_temp_seq'], dtype=np.float32)
        outdoor_comparison = np.asarray(comparison_entry['outdoor_temp_seq'], dtype=np.float32)
        local_on_primary = compute_local_on(local_primary, outdoor_primary)
        local_on_comparison = compute_local_on(local_comparison, outdoor_comparison)
        matches, comps = count_agreement(local_on_primary, local_on_comparison)
        local_matches += matches
        local_comparisons += comps

        exec_changes = np.diff(win_comparison[:min_len], prepend=win_comparison[0])
        ref_changes = np.diff(win_primary[:min_len], prepend=win_primary[0])
        if exec_changes.size > 1 and np.var(exec_changes) > 0 and np.var(ref_changes) > 0:
            window_correlations.append(float(np.corrcoef(exec_changes, ref_changes)[0, 1]))

    overall_window_agreement = window_matches / window_comparisons if window_comparisons else float('nan')
    overall_fcu_agreement = fcu_matches / fcu_comparisons if fcu_comparisons else float('nan')
    overall_local_agreement = local_matches / local_comparisons if local_comparisons else float('nan')
    overall_window_corr = float(np.nanmean(window_correlations)) if window_correlations else float('nan')

    print(f"Overall window agreement ({COMPARISON_POLICY_NAME} vs {PRIMARY_POLICY_NAME}): {overall_window_agreement:.3%}")
    print(f"Overall FCU on/off agreement: {overall_fcu_agreement:.3%}")
    print(f"Overall local cooling on/off agreement: {overall_local_agreement:.3%}")
    print(f"Overall window action correlation: {overall_window_corr:.3f}")


In [ ]:

import numpy as np

def _compute_violation_pct(row, ceiling=30.0):
    zone_seq = row.get('zone_temp_seq', [])
    if not zone_seq:
        return float('nan')
    arr = np.asarray(zone_seq, dtype=np.float32)
    if arr.size == 0:
        return float('nan')
    if arr.ndim == 1:
        arr = arr[:, None]
    avg_temp = arr.mean(axis=1)
    violations = avg_temp > ceiling
    return float(violations.mean() * 100.0)

if 'results_df' not in locals():
    raise RuntimeError('results_df missing. Run evaluation cells first.')

results_df['avg_energy_per_hour'] = results_df['energy_kwh'] / np.maximum(results_df['episode_length'], 1) * 60.0
results_df['avg_window_switches'] = results_df['window_switches'] / np.maximum(results_df['episode_length'] / 60.0, 1e-6)
results_df['violation_minutes_pct'] = results_df.apply(_compute_violation_pct, axis=1)



In [ ]:
if 'results_df' not in locals():
    raise RuntimeError('results_df missing. Run evaluation cells first.')

available_labels = [PRIMARY_EXEC_LABEL]
if HAS_COMPARISON_POLICY:
    available_labels.append(COMPARISON_EXEC_LABEL)

summary = (
    results_df[results_df['policy_label'].isin(available_labels)]
    [['policy_label', 'avg_energy_per_hour', 'violation_minutes_pct', 'avg_window_switches']]
    .groupby('policy_label')
    .agg(['mean', 'std'])
)
print('Policy comparison (means ± std):' if HAS_COMPARISON_POLICY else 'Policy metrics (means ± std):')
display(summary)
